In [19]:
# Core
import os
from dotenv import load_dotenv

In [35]:
# Libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [21]:
# Setup API Key
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

In [22]:
# Load Files
embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # Specific embedding model
llm = ChatOpenAI(
  model_name='gpt-3.5-turbo',
  max_tokens=500,
)

pdf_path = 'rag_file_example.pdf'
loader = PyPDFLoader(pdf_path, extract_images=False)
pages = loader.load_and_split() # PDF File chunked

In [23]:
# Splitters
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

In [24]:
# Storages (child - chromaDb / parent - in memory)
store = InMemoryStore()
vector_store = Chroma(embedding_function=embeddings, persist_directory='child_vector_db')

In [ ]:
# Create Retriever
parent_document_retriever = ParentDocumentRetriever(
  vectorstore=vector_store, # Children
  docstore=store, # Parents
  child_splitter=child_splitter,
  parent_splitter=parent_splitter,
)

parent_document_retriever.add_documents(pages, ids=None) # Add pages and execute the retriever

In [33]:
# Start Template
TEMPLATE = """
    Voce é um especialista em legislação e tecnologia, responda a pergunta abaixo ultilizando o contexto informado.
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [37]:
# Build Chain
setup_retrieval = RunnableParallel({'question': RunnablePassthrough(), 'context': parent_document_retriever})
output_parser = StrOutputParser()

parent_chain_retrieval = setup_retrieval | rag_prompt | llm | output_parser

In [38]:
parent_chain_retrieval.invoke('Quais os principais pontos que devo prestar atenção sobre o marco da AI?')

'Os principais pontos que você deve prestar atenção sobre o marco da AI são:\n\n1. Categorização dos riscos da inteligência artificial: é importante avaliar se o sistema de AI é de alto risco ou categorizado como de risco excessivo, pois isso determinará as normas de controle a serem seguidas.\n   \n2. Governança dos sistemas: é fundamental adotar medidas para garantir transparência e mitigação de vieses nos sistemas de AI, especialmente para sistemas de alto risco e sistemas governamentais de inteligência artificial.\n   \n3. Regras de responsabilização civil: é essencial entender as hipóteses em que os responsáveis pelo desenvolvimento e utilização de sistemas de AI não serão responsabilizados, bem como as medidas de responsabilização para sistemas de alto risco ou de risco excessivo.\n\n4. Proteção contra discriminação: o projeto reforça a proteção contra discriminação, incluindo direitos como informação, contestação e correção de vieses discriminatórios, além de medidas de governan